# Análisis de Ratings Elo - Mundial 2026

In [8]:
import pandas as pd
import numpy as np
from collections import defaultdict
import matplotlib.pyplot as plt
import os

os.makedirs('output', exist_ok=True)

## Cargar datos

In [9]:
matches = pd.read_csv('Data/ultimas_clasificatorias.csv')
matches['date'] = pd.to_datetime(matches['date'])
print(f"Partidos: {len(matches)}")
print(f"Fechas: {matches['date'].min()} a {matches['date'].max()}")

Partidos: 1001
Fechas: 2022-01-27 00:00:00 a 2026-03-31 00:00:00


## Calcular Elo (K variable)

In [10]:
def calculate_elo(matches_df, initial_rating=1500, k=32, home_adv=100, k_variable=True):
    ratings = defaultdict(lambda: initial_rating)
    history = []
    n_matches = len(matches_df)
    
    for i, (_, row) in enumerate(matches_df.iterrows()):
        # K variable: 20% primeros con K doble, luego baja gradualmente
        if k_variable:
            if i < n_matches * 0.2:
                current_k = k * 2  # K doble al inicio
            else:
                progress = (i - n_matches * 0.2) / (n_matches * 0.8)
                current_k = k * (2 - progress)  # De 2K a K
        else:
            current_k = k
        
        home = row['home_team']
        away = row['away_team']
        home_goals = row['home_score']
        away_goals = row['away_score']

        r_home = ratings[home]
        r_away = ratings[away]

        e_home = 1 / (1 + 10 ** ((r_away - r_home - home_adv) / 400))
        e_away = 1 / (1 + 10 ** ((r_home + home_adv - r_away) / 400))

        if home_goals > away_goals:
            s_home, s_away = 1, 0
        elif home_goals < away_goals:
            s_home, s_away = 0, 1
        else:
            s_home, s_away = 0.5, 0.5

        ratings[home] += current_k * (s_home - e_home)
        ratings[away] += current_k * (s_away - e_away)

        history.append({'date': row['date'], 'team': home, 'rating': ratings[home], 'k_used': current_k})
        history.append({'date': row['date'], 'team': away, 'rating': ratings[away], 'k_used': current_k})

    return dict(ratings), pd.DataFrame(history)

ratings, elo_history = calculate_elo(matches)
print(f"Equipos únicos: {len(ratings)}")

Equipos únicos: 209


In [11]:
# Ver cómo varía K
print("K usado en distintos puntos:")
print(f"Al inicio (0%): {elo_history['k_used'].iloc[0]:.1f}")
print(f"Al 20%: {elo_history['k_used'].iloc[len(elo_history)//5]:.1f}")
print(f"Al final: {elo_history['k_used'].iloc[-1]:.1f}")

K usado en distintos puntos:
Al inicio (0%): 64.0
Al 20%: 64.0
Al final: 32.0


## Top 20 - Elo de Campeonato

In [12]:
ratings_df = pd.DataFrame(list(ratings.items()), columns=['Equipo', 'Rating'])
ratings_df = ratings_df.sort_values('Rating', ascending=False).reset_index(drop=True)
print("=== Top 20 Equipos - Elo Campeonato ===\n")
print(ratings_df.head(20).to_string())

=== Top 20 Equipos - Elo Campeonato ===

         Equipo       Rating
0   South Korea  1729.275293
1         Japan  1725.123987
2          Iran  1709.685270
3    Uzbekistan  1693.193256
4     Australia  1690.445185
5       Tunisia  1677.609169
6    Costa Rica  1672.566124
7          Iraq  1666.061419
8   New Zealand  1664.628110
9       Morocco  1654.345848
10  Ivory Coast  1651.100402
11        Egypt  1647.569524
12    Argentina  1637.472461
13      England  1636.818496
14       Norway  1636.660568
15      Senegal  1633.199368
16      Curaçao  1629.807654
17        Ghana  1625.048129
18      Algeria  1623.809597
19       Jordan  1623.540897


## Elo de Shootouts (Penales)

In [13]:
shootouts = pd.read_csv('Data/shootouts.csv')
shootouts['date'] = pd.to_datetime(shootouts['date'])
shootouts = shootouts[shootouts['date'] >= pd.to_datetime('2022-06-01')]
print(f"Shootouts desde junio 2022: {len(shootouts)}")

Shootouts desde junio 2022: 88


In [14]:
def calculate_shootout_elo(shootouts_df, initial_rating=1500, k=20):
    ratings = defaultdict(lambda: initial_rating)

    for _, row in shootouts_df.iterrows():
        winner = row['winner']
        loser = row['loser']
        r_winner = ratings[winner]
        r_loser = ratings[loser]
        e_winner = 1 / (1 + 10 ** ((r_loser - r_winner) / 400))
        ratings[winner] += k * (1 - e_winner)
        ratings[loser] += k * (0 - (1 - e_winner))

    return dict(ratings)

shootout_ratings = calculate_shootout_elo(shootouts)
shootout_df = pd.DataFrame(list(shootout_ratings.items()), columns=['Equipo', 'Shootout Elo'])
shootout_df = shootout_df.sort_values('Shootout Elo', ascending=False).reset_index(drop=True)
print("=== Top 20 Equipos - Elo Penales ===\n")
print(shootout_df.head(20).to_string())

KeyError: 'loser'

## Comparación

In [ ]:
combined = ratings_df.merge(shootout_df, on='Equipo', how='outer').fillna(1500)
combined['Diff'] = combined['Rating'] - combined['Shootout Elo']
combined = combined.sort_values('Rating', ascending=False).reset_index(drop=True)
print("=== Comparación: Campeonato vs Penales ===\n")
print(combined.head(20).to_string())

## Evolución - Un Equipo

In [ ]:
team = 'Argentina'
team_history = elo_history[elo_history['team'] == team].sort_values('date')

plt.figure(figsize=(12, 5))
plt.plot(team_history['date'], team_history['rating'], marker='.', linewidth=1)
plt.title(f'Evolución Elo - {team}')
plt.xlabel('Fecha')
plt.ylabel('Rating')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Evolución - Varios Equipos

In [ ]:
teams = ['Argentina', 'Brazil', 'France', 'Germany', 'Spain', 'England']

plt.figure(figsize=(14, 6))
for team in teams:
    team_hist = elo_history[elo_history['team'] == team].sort_values('date')
    plt.plot(team_hist['date'], team_hist['rating'], marker='.', label=team, linewidth=1.5)

plt.title('Evolución Elo - Principales Candidatos')
plt.xlabel('Fecha')
plt.ylabel('Rating')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Guardar CSVs

In [ ]:
ratings_df.to_csv('output/ratings_championship.csv', index=False)
shootout_df.to_csv('output/ratings_shootout.csv', index=False)
combined.to_csv('output/ratings_combined.csv', index=False)
elo_history.to_csv('output/elo_history.csv', index=False)
print("Archivos guardados en output/")